# Creating simulated datasets

This is step 1 of the ISO end-to-end pipeline:

1. **this notebook** synthesises the datasets,
2. [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) builds the cross-covariance
   between them,
3. [`analyse_datasets.ipynb`](analyse_datasets.ipynb) runs the joint fit on exactly these products.

All artifacts go to a stable, git-ignored `sims/` directory that the downstream notebooks read by
the same filenames. We build, from simplest to most involved:

1. **From scratch** — a galaxy x CMB-lensing cross-correlation, computed with
   [CCL](https://github.com/LSSTDESC/CCL), with the cosmology and the noise exposed as knobs.
2. **A smooth CMB-lensing twin** — the shipped reconstruction with its bandpowers replaced by the
   theory at the ISO fiducial (chi-square = 0 when fit at the same cosmology), via
   `soliket.sacc_tools.smooth_twin_sacc`.
3. **A smooth MFLike CMB+foregrounds dataset** — the per-frequency theory binned through MFLike's own
   windows, via `soliket.sacc_tools.smooth_mflike_sacc`.

Datasets 2 and 3 are the joint-analysis inputs; everything is built at the **ISO fiducial**
(`defaults_dir="defaults"` — cosmology incl. the SO normal-hierarchy neutrinos).

In [ ]:
from pathlib import Path

import numpy as np
import sacc

# All three ISO notebooks share this directory under the same name (git-ignored).
SIMS = Path("sims")
SIMS.mkdir(exist_ok=True)

# Canonical artifact filenames the downstream notebooks read.
GALAXY_KAPPA = SIMS / "galaxy_kappa.sim.fits"
LENSING_SMOOTH = SIMS / "lensing_smooth.sacc.fits"
MFLIKE_SMOOTH = SIMS / "mflike_smooth.fits"
print("writing simulated datasets to", SIMS.resolve())

## 1. A galaxy x kappa dataset from scratch

We simulate the cross-correlation of an unWISE-like galaxy sample with the SO CMB-lensing
convergence. The three spectra (`gg`, `gk`, `kk`) come from CCL; the bandpower windows and the joint
covariance come from `soliket.sacc_tools`. The **cosmology is sourced from the same shared ISO
fiducial** as the lensing / MFLike twins (`defaults/cosmo.yaml`, translated to CCL convention by
`iso_ccl_cosmology`), so all three datasets agree on the cosmological parameters. We wrap the build in
a function so the **cosmology** and the **galaxy noise spectrum** stay explicit knobs, then build the
canonical dataset at the fiducial and two variants (a lower-amplitude cosmology, a custom noise).

In [ ]:
import pyccl as ccl

from soliket.presets import load_fiducial_map
from soliket.sacc_tools import gaussian_covariance, top_hat_windows

# Fiducial galaxy redshift distribution, borrowed from the shipped reference dataset.
ref = sacc.Sacc.load_fits("../../tests/data/unwise_g-so_kappa.sim.sacc.fits")
Z, NZ = ref.tracers["gc_unwise"].z, ref.tracers["gc_unwise"].nz


def iso_ccl_cosmology(defaults_dir="defaults", **overrides):
    """A ``ccl.Cosmology`` at the ISO fiducial, read from ``defaults/cosmo.yaml``.

    The galaxy x kappa dataset uses the SAME shared fiducial as the lensing / MFLike
    twins, translated from CAMB convention (``H0``, ``ombh2``, ``omch2``, ``logA``)
    to CCL's (``h``, ``Omega_b``, ``Omega_c``, ``A_s``) -- so all three datasets share
    one set of cosmological parameters instead of an ad-hoc one. `overrides` replace
    individual CCL kwargs (e.g. ``A_s=1.8e-9`` for the low-amplitude knob), keeping the
    cosmology a knob while anchoring its baseline to the shared fiducial.

    Massive neutrinos are omitted here: the ISO sum (~0.06 eV) is a sub-percent effect
    at l <= 600 for this standalone demo, and the CCL/CAMB neutrino conventions differ
    (CCL's normal-hierarchy minimum exceeds the ISO sum), so exact matching is not
    meaningful at this level.
    """
    cosmo = load_fiducial_map(defaults_dir)["cosmo"]

    def central(name):  # central value of a cosmo.yaml param spec
        spec = cosmo[name]
        return spec["ref"]["loc"] if "ref" in spec else spec["value"]

    h = central("H0") / 100.0
    kwargs = dict(
        h=h,
        Omega_b=central("ombh2") / h**2,
        Omega_c=central("omch2") / h**2,
        n_s=central("ns"),
        A_s=1e-10 * np.exp(central("logA")),  # As(logA), per cosmo.yaml
        matter_power_spectrum="linear",
    )
    kwargs.update(overrides)
    return ccl.Cosmology(**kwargs)


def build_galaxy_kappa(
    out_path,
    *,
    cosmo,
    ngal_per_arcmin2=1.0,
    fsky=0.4,
    ell_max=600,
    n_bins=20,
    noise_gg=None,
):
    """Simulate an unWISE-like galaxy x SO CMB-lensing cross-correlation SACC.

    `cosmo` is a ``ccl.Cosmology``; `noise_gg` optionally overrides the galaxy
    auto-spectrum noise (default: shot noise ``1 / n_gal``). Returns the SACC.
    """
    b1, mag_bias = 1.0, 0.4
    gc = ccl.NumberCountsTracer(
        cosmo,
        has_rsd=False,
        dndz=(Z, NZ),
        bias=(Z, b1 * np.ones_like(Z)),
        mag_bias=(Z, mag_bias * np.ones_like(Z)),
    )
    ck = ccl.CMBLensingTracer(cosmo, z_source=1086.0)

    ells, window = top_hat_windows(ell_max, n_bins)
    # Bin theory through the window rather than sampling it at the bin centres, so
    # the stored data is exactly what the likelihood computes for it (`w_bins @ cl`,
    # a plain contraction). Sampling at centres instead leaves a curvature residual
    # -- the bin mean of a curved spectrum is not its centre value -- worth ~13% in
    # the first bin, where C_ell is steepest across the bin's 30 multipoles.
    w_bins = window.weight.T
    support = np.asarray(window.values, dtype=float)
    # Per-bin widths read off the window, not assumed uniform: top_hat_windows
    # splits ell_max + 1 multipoles into n_bins, so when that does not divide
    # evenly the leading bins are one multipole wider, and Knox goes as 1/delta_ell.
    delta_ell = (w_bins != 0).sum(axis=1)
    cl_gg = w_bins @ ccl.angular_cl(cosmo, gc, gc, support)
    cl_gk = w_bins @ ccl.angular_cl(cosmo, gc, ck, support)
    cl_kk = w_bins @ ccl.angular_cl(cosmo, ck, ck, support)

    if noise_gg is None:  # default: galaxy shot noise 1 / n_gal
        ngal_sr = ngal_per_arcmin2 / np.deg2rad(1.0 / 60.0) ** 2
        noise_gg = np.full_like(cl_gg, 1.0 / ngal_sr)
    cls = np.array([[cl_gg + noise_gg, cl_gk], [cl_gk, cl_kk]])
    cov = gaussian_covariance(cls, ells, delta_ell, fsky)

    s = sacc.Sacc()
    s.metadata["info"] = "Simulated unWISE-like galaxy x SO CMB-lensing cross-correlation"
    s.add_tracer(
        "NZ",
        "gc_unwise",
        quantity="galaxy_density",
        spin=0,
        z=Z,
        nz=NZ,
        metadata={"ngal": ngal_per_arcmin2},
    )
    s.add_tracer(
        "Map",
        "ck_so",
        quantity="cmb_convergence",
        spin=0,
        ell=np.arange(3000),
        beam=np.ones(3000),
    )
    s.add_ell_cl("cl_00", "gc_unwise", "gc_unwise", ells, cl_gg, window=window)
    s.add_ell_cl("cl_00", "gc_unwise", "ck_so", ells, cl_gk, window=window)
    s.add_ell_cl("cl_00", "ck_so", "ck_so", ells, cl_kk, window=window)
    s.add_covariance(cov)
    s.save_fits(str(out_path), overwrite=True)
    return s

In [ ]:
# Canonical dataset at the shared ISO fiducial cosmology (from defaults/cosmo.yaml).
fiducial = iso_ccl_cosmology()
s = build_galaxy_kappa(GALAXY_KAPPA, cosmo=fiducial)
print("wrote", GALAXY_KAPPA.name, "->", len(s.mean), "data points")

# Knob 1 - a lower-amplitude deviation from the fiducial (lower A_s -> lower sigma8).
low_amp = iso_ccl_cosmology(A_s=1.8e-9)
build_galaxy_kappa(SIMS / "galaxy_kappa.lowA.fits", cosmo=low_amp)

# Knob 2 - a custom (flat, noisier) galaxy noise spectrum instead of pure shot noise.
ells, _ = top_hat_windows(600, 20)
build_galaxy_kappa(
    SIMS / "galaxy_kappa.noisy.fits", cosmo=fiducial, noise_gg=np.full(len(ells), 5e-7)
)
print("knob variants written (lower-amplitude cosmology, custom noise)")

### Eyeball the build and its knobs

Before handing these files downstream, look at what was written. First the galaxy
redshift kernel `n(z)` that drives `gg`/`gk`, then the three spectra (`gg`, `gk`,
`kk`) for the fiducial build and the two knob variants. The knobs become visible
here: **low $A_s$** pulls `gk`/`kk` down (lower $\sigma_8$), while the **custom noise**
inflates the `gg` errors and raises its floor, leaving `gk`/`kk` on the fiducial.

In [ ]:
import matplotlib.pyplot as plt


def read_cl(path, t1, t2):
    """(ell, C_ell, err) for a cl_00 spectrum in a SACC file; err is None if no cov."""
    s = sacc.Sacc.load_fits(str(path))
    try:
        ell, cl, cov = s.get_ell_cl("cl_00", t1, t2, return_cov=True)
        return ell, cl, np.sqrt(np.diag(cov))
    except ValueError:  # e.g. sims that carry only bandpowers
        ell, cl = s.get_ell_cl("cl_00", t1, t2, return_cov=False)
        return ell, cl, None


# The galaxy redshift kernel driving gg / gk (context for the spectra below).
plt.figure(figsize=(6, 3))
plt.fill_between(Z, NZ, alpha=0.4)
plt.plot(Z, NZ)
plt.xlabel("$z$"); plt.ylabel("$n(z)$"); plt.title("unWISE-like galaxy sample")
plt.tight_layout(); plt.show()

# What the knobs did: the fiducial build vs the two variants written above, with
# error bars from each SACC's covariance.
builds = {
    "fiducial": GALAXY_KAPPA,
    "low $A_s$": SIMS / "galaxy_kappa.lowA.fits",
    "noisy $gg$": SIMS / "galaxy_kappa.noisy.fits",
}
specs = [
    ("gc_unwise", "gc_unwise", r"$gg$"),
    ("gc_unwise", "ck_so", r"$g\kappa$"),
    ("ck_so", "ck_so", r"$\kappa\kappa$"),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (t1, t2, lab) in zip(axes, specs):
    for name, path in builds.items():
        ell, cl, err = read_cl(path, t1, t2)
        ax.errorbar(ell, cl, err, fmt="o", ms=3, capsize=2, label=name)
    ax.set(title=lab, xlabel=r"$\ell$", yscale="log")
    ax.legend(fontsize="small")
axes[0].set_ylabel(r"$C_\ell$")
fig.tight_layout(); plt.show()

## 2. A smooth CMB-lensing twin

A *smooth* twin reuses the shipped dataset's tracers, bandpower windows and covariance but replaces
the noisy measured bandpowers with the theory at a chosen cosmology — so the likelihood gives
chi-square = 0 when fit at that cosmology. `soliket.sacc_tools.smooth_twin_sacc` does the SACC
surgery; we get the binned theory from the evaluated `lensing` component **of the joint
`multigaussian` model** (via `resolve_aliases`, the named role). Imprinting from the *same* model the
analysis fits with matters here: the joint CAMB is driven by both members' requirements, so a
standalone-`lensing` imprint would leave a small (~0.01) residual chi-square — pulling the lensing
component out of the joint model makes it exactly zero. The **imprint cosmology is a knob**: by
default the ISO fiducial, but any param override (e.g. a shifted `tau`) imprints a non-fiducial twin
for parameter-recovery tests.

> **Overriding likelihood options per folder.** Beyond the param/`theory.yaml` overrides in
> `defaults/`, you can patch a preset's *likelihood/theory skeleton* by dropping a
> `defaults/templates/<preset>.yaml` — it is layered onto the packaged template (`recursive_update`,
> last wins), so you set only the keys you want (e.g. `theory_lmax`). Because the `multigaussian`
> preset *composes* its members, an override to `templates/lensing.yaml` reaches **both** this
> standalone lensing build and the joint fit — imprint and fit stay consistent by construction. We
> keep the ISO `defaults/` at the packaged values here, so no such file is shipped.

In [ ]:
from cobaya.model import get_model
from cobaya.tools import resolve_packages_path

from soliket.presets import build_info, resolve_aliases
from soliket.sacc_tools import smooth_twin_sacc


def smooth_lensing(out_path, **param_overrides):
    """Write a smooth (theory) CMB-lensing twin, imprinted from the JOINT model.

    The twin is fit by the ``multigaussian`` preset (analyse_datasets.ipynb), whose
    shared CAMB is driven by *both* members' requirements. Imprinting from that same
    joint model -- rather than the standalone ``lensing`` preset -- makes the binned
    clkk bit-identical to what the fit recomputes, so the joint chi-square is 0
    *exactly*. (A standalone-lensing imprint leaves a ~0.01 residual: the lensing-only
    CAMB precision differs from the joint, and matching extra_args by hand does not
    close it -- only the same model does.) `param_overrides` pins fiducial params
    (e.g. ``tau=0.06``) to imprint a non-fiducial twin. Returns
    ``(lensing_likelihood, binned_clkk)``. Note: builds the full joint model, so it
    needs the MFLike data and runs CAMB at the joint accuracy (a few minutes).
    """
    info = build_info("multigaussian", defaults_dir="defaults")
    info["packages_path"] = resolve_packages_path()
    for name, value in param_overrides.items():
        info["params"][name] = {"value": value}

    model = get_model(info)
    model.loglikes({})  # evaluate at the imprint cosmology
    lensing = resolve_aliases(model).lensing
    clkk = lensing._get_theory()  # binned C_ell^kappakappa

    src = sacc.Sacc.load_fits(lensing.datapath)  # reuse shipped tracers/windows/cov
    smooth_twin_sacc(src, "cl_00", "ck", "ck", clkk, out_path=out_path)
    return lensing, clkk


lensing, clkk = smooth_lensing(LENSING_SMOOTH)
print("wrote", LENSING_SMOOTH.name, "->", len(clkk), "bins")

In [ ]:
# Knob - imprint a second twin at a shifted tau (a parameter-recovery target).
smooth_lensing(SIMS / "lensing_smooth.tau0p06.sacc.fits", tau=0.06)
print("wrote a tau=0.06 twin alongside")

### What "smooth twin" means

The smooth twin reuses the shipped windows and covariance but replaces the noisy
bandpowers with theory, so it lies as a clean curve through the data it was carved
from — that is the χ²=0 statement, made visual.

We built a second twin at `tau=0.06` above, but it is **not** overlaid here: $C_L^{\kappa\kappa}$
is τ-insensitive (it shifts by <0.03%, invisible on this scale), because lensing is
not reionization-suppressed. That is precisely why the τ-recovery test needs the
primary-CMB member of the joint fit, not lensing alone.

In [ ]:
# Smooth twin (theory, no scatter) over the shipped noisy reconstruction it was
# carved from: the data scatters around the twin within its errors -> chi2 = 0 at the
# imprint cosmology.
fig, ax = plt.subplots(figsize=(7, 4.5))
L, cl, err = read_cl(lensing.datapath, "ck", "ck")  # shipped SACC (has covariance)
ax.errorbar(L, cl, err, fmt="o", ms=4, color="brown", label="shipped (noisy)")
L, cl, _ = read_cl(LENSING_SMOOTH, "ck", "ck")
ax.plot(L, cl, "-", color="tab:blue", lw=2, label="smooth twin")
ax.set(xlabel="$L$", ylabel=r"$C_L^{\kappa\kappa}$", title="CMB lensing", xscale="log")
ax.legend(); fig.tight_layout(); plt.show()

## 3. A smooth MFLike CMB + foregrounds dataset

The same smooth-twin idea for the primary CMB, where the data vector spans many frequency
cross-spectra. The per-frequency plumbing — combining CMB + foregrounds + systematics through
MFLike's `get_modified_theory`, binning with MFLike's own bandpower windows, and writing one `NuMap`
tracer per `(frequency, spin)` channel — lives in `soliket.sacc_tools.smooth_mflike_sacc`, which
takes the concrete handles (the evaluated likelihood + theory outputs, not a `Session`).

This build runs CAMB at MFLike accuracy and takes a few minutes; the covariance and bandpower-window
(Bbl) matrices are reused from the shipped `cov_Bbl_file`, so only the data vector is regenerated.

In [ ]:
from soliket.sacc_tools import smooth_mflike_sacc

RUN_MFLIKE = True  # set False to skip the few-minute CAMB build

if RUN_MFLIKE:
    info = build_info("mflike", defaults_dir="defaults")
    info["packages_path"] = resolve_packages_path()
    model = get_model(info)
    roles = resolve_aliases(model)

    # Numeric fiducial values (skip lambda-valued / derived params).
    params = {
        k: v["value"]
        for k, v in info["params"].items()
        if isinstance(v, dict) and "value" in v and not isinstance(v["value"], str)
    }
    model.loglikes(params)  # evaluate at the ISO fiducial

    dls = model.provider.get_Cl(ell_factor=True)
    fg_totals = roles.foreground.get_fg_totals()
    smooth_mflike_sacc(roles.mflike, dls, fg_totals, params, out_path=MFLIKE_SMOOTH)
    print("wrote", MFLIKE_SMOOTH.name)
else:
    print("RUN_MFLIKE is False - skipping the smooth MFLike build.")

### Smooth CMB twin vs. a noisy sim

Same idea for the primary CMB. The smooth twin is the binned theory the analysis
fits; overlaid on a noisy MFLike realization (`LAT_simu_sacc_00000`) for one channel
(TT, 93×93), the noisy bandpowers scatter around it — acoustic peaks at low ℓ, the
foreground-dominated rise at high ℓ.

In [ ]:
# Smooth CMB twin (binned theory the analysis fits) over a noisy MFLike sim, for one
# channel (TT, 93x93). The noisy bandpowers scatter around the smooth curve. (The
# MFLike covariance lives in the separate cov_Bbl file, so we overlay bandpowers only.)
if MFLIKE_SMOOTH.exists():
    noisy = Path(resolve_packages_path()) / "data/MFLike/v0.8/LAT_simu_sacc_00000.fits"
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ell, cl, _ = read_cl(noisy, "LAT_93_s0", "LAT_93_s0")
    ax.plot(ell, cl, ".", ms=4, alpha=0.45, color="grey", label="noisy sim 00000")
    ell, cl, _ = read_cl(MFLIKE_SMOOTH, "LAT_93_s0", "LAT_93_s0")
    ax.plot(ell, cl, "-", color="tab:red", lw=1.5, label="smooth twin")
    ax.set(xlabel=r"$\ell$", ylabel=r"$D_\ell^{TT}\ [\mu K^2]$",
           title=r"MFLike TT 93$\times$93: smooth twin vs noisy sim")
    ax.legend(); fig.tight_layout(); plt.show()
else:
    print("mflike_smooth.fits not found - run the build cell above with RUN_MFLIKE=True.")

## BYOB — Bring Your Own Bandpowers

The smooth twins above imprint the model's *own* theory; only the cosmology is a knob.
The other direction — pointing a lensing likelihood at a spectrum **you** bring (a sim,
or an external CAMB run) — is one short recipe: bin your `C_L^kk` through bandpower
windows, assemble a SACC with `build_lensing_sacc`, and fit. With the correction-free
`LensingLite` the stored bandpowers *are* the binned theory, so χ²=0 at that cosmology.

Standalone by design: written to `sims/byob/`, not consumed downstream, so the pipeline's
χ²=0 claim is untouched.

> Using the **full** `LensingLikelihood` instead? It also needs N0/N1 estimator-bias aux
> files. If you have your own (e.g. from so-lenspipe), `build_lensing_corrections_sacc`
> writes them in the layout it reads — see `tests/test_sacc_tools.py` for a round-trip.

In [ ]:
from soliket.lensing.lensing import LensingLikelihood
from soliket.sacc_tools import build_lensing_sacc

BYOB = SIMS / "byob"
BYOB.mkdir(exist_ok=True)
LMAX = 2000
# BYOB is standalone, so any fixed cosmology works; here a concrete lensing fiducial.
byob_cosmo = dict(LensingLikelihood.fiducial_params)

# 1. Your own C_L^kk. A standalone CAMB run stands in for "your spectrum" — swap in a
#    sim or any external computation sampled on integer multipoles 0..LMAX.
info_cl = {"params": byob_cosmo,
           "likelihood": {"soliket.utils.OneWithCls": {"lmax": LMAX}},
           "theory": {"camb": {"extra_args": {"lens_potential_accuracy": 1}}},
           "packages_path": resolve_packages_path()}
m_cl = get_model(info_cl)
m_cl.logposterior({})
ls = np.arange(LMAX)
# Here use your C_L^kk --->
clkk = (ls * (ls + 1)) ** 2 * m_cl.provider.get_Cl(ell_factor=False)["pp"][:LMAX] * 0.25

# 2. Bin through your own top-hat windows; a signal-only Knox covariance for the demo.
ell, win = top_hat_windows(ell_max=LMAX - 1, n_bins=20)
w_bins = win.weight.T
binned = w_bins @ clkk[np.asarray(win.values, int)]
cov = gaussian_covariance([[binned]], ell, (w_bins != 0).sum(axis=1), fsky=0.4)

# 3. Assemble the data SACC; 4. fit with the correction-free LensingLite -> chi2 = 0,
#    because the stored bandpowers ARE the binned theory the likelihood recomputes.
byob_data = BYOB / "lensing_byob.sacc.fits"
build_lensing_sacc(ell, binned, cov, windows=win, out_path=byob_data)
info_fit = {"params": byob_cosmo,
            "likelihood": {"soliket.LensingLiteLikelihood":
                           {"datapath": str(byob_data), "lmax": LMAX}},
            "theory": {"camb": {"extra_args": {"lens_potential_accuracy": 1}}},
            "packages_path": resolve_packages_path()}
m_fit = get_model(info_fit)
lite = m_fit.likelihood["soliket.LensingLiteLikelihood"]
chi2 = -2 * (float(m_fit.loglikes(byob_cosmo)[0].sum()) - lite.data.norm_const)
print(f"wrote {byob_data.name}; LensingLite chi2 = {chi2:.3g}  (0 => self-consistent)")

## Recap

`sims/` now holds the pipeline inputs:

| Artifact | Built from | Consumed by |
| --- | --- | --- |
| `mflike_smooth.fits` | `smooth_mflike_sacc` (CMB+fg theory, ISO fiducial) | the analysis (MFLike component) |
| `lensing_smooth.sacc.fits` | `smooth_twin_sacc` (lensing theory, ISO fiducial) | the analysis (lensing component) |
| `galaxy_kappa.sim.fits` (+ knob variants) | CCL from scratch | standalone galaxy x kappa example |

The smooth lensing twin carries only the data + covariance; the analysis notebook points the
likelihood's `correction_filename` / `fiducial_filename` at the shipped (multi-hundred-MB) auxiliary
files in place, so we never copy those into `sims/`. Both smooth twins are built at the **same ISO
fiducial** the analysis fits at, so the joint chi-square is 0 by construction.

Next: [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) builds the cross-covariance
between the MFLike and lensing data vectors; then
[`analyse_datasets.ipynb`](analyse_datasets.ipynb) runs the joint fit on these `sims/` products.